# 01 — Sanity check dos dados brutos do ALTA

**Objetivo.** Validar a integridade e a qualidade de `dados/brutos/` **antes** de qualquer EDA, blocagem ou anotação. É uma ferramenta de diagnóstico interna: não gera tabela nem figura para `latex/tcc_artigo/`.

**Regras deste notebook**
- Somente leitura. Nada é limpo, corrigido ou gravado; o que precisar de correção aparece no veredicto final.
- Todos os números e exemplos vêm dos dados reais. Amostragens usam `SEED = 42`.
- A leitura reaproveita as funções de `scripts/01_coleta_dados.py`, sem duplicar a lógica.
- Papéis das colunas conforme o **ADR-0011**: o modelo só vê o texto (`description`, `desc_alim`). As colunas do matcher ALTA (`codg_item`, `desc_ref`, `match_score`) aparecem aqui apenas em agregados estruturais (ADR-0011, uso (d)); nenhuma linha delas é exibida.

**Critério de gravidade**
| Nível | Quando | Consequência |
|---|---|---|
| `BLOQUEANTE` | compromete a entrada do modelo (texto), a identificação de registros ou o dicionário, ou quebra o escopo (PB/CE/PI) | corrigir antes da EDA/blocagem |
| `WARNING` | defeito real, mas que não impede o matching; afeta metadado, baseline ou generalização | seguir, e citar no TCC como limitação dos dados |
| `INFO` | caracterização sem defeito | só registrar |
| `PASSOU` | verificação sem achado | — |

In [1]:
import importlib.util
import re
import unicodedata
from pathlib import Path

import pandas as pd

SEED = 42
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 140)

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DADOS_BRUTOS = RAIZ / "dados" / "brutos"

# Reaproveita o carregamento/validação do script 01 (nome numérico: import por caminho).
_spec = importlib.util.spec_from_file_location("coleta_dados", RAIZ / "scripts" / "01_coleta_dados.py")
coleta = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(coleta)

VEREDITOS: list[dict] = []


def registrar(nivel: str, secao: str, mensagem: str) -> None:
    assert nivel in {"BLOQUEANTE", "WARNING", "INFO", "PASSOU"}
    VEREDITOS.append({"nivel": nivel, "secao": secao, "mensagem": mensagem})
    print(f"[{nivel}] {mensagem}")


def normalizar(texto) -> str:
    sem_acento = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    return sem_acento.upper().strip()

## 1. Carregamento

Usa `localizar_arquivo_mercados`, `validar_produtos_coletados` e `validar_dicionario_canonico` do script 01. Essas funções já falham com erro claro se um arquivo estiver ausente ou sem as colunas mínimas — se alguma lançar exceção, o veredicto é `BLOQUEANTE` por definição.

In [2]:
SECAO = "1. Carregamento"
caminho_mercados = coleta.localizar_arquivo_mercados(DADOS_BRUTOS)
df = coleta.validar_produtos_coletados(caminho_mercados)
dic = coleta.validar_dicionario_canonico(DADOS_BRUTOS / "dicionario_canonico.csv")

registrar("PASSOU", SECAO, f"Base carregada: {caminho_mercados.name} ({len(df):,} linhas × {df.shape[1]} colunas).")
registrar("PASSOU", SECAO, f"Dicionário carregado ({len(dic):,} linhas × {dic.shape[1]} colunas lidas).")

2026-10-06 12:08:19,183 INFO Produtos coletados: todos_mercados_2026-09-10.csv (30259 linhas, 13 colunas)


2026-10-06 12:08:19,185 INFO Dicionário canônico: dicionario_canonico.csv (1593 entradas)


[PASSOU] Base carregada: todos_mercados_2026-09-10.csv (30,259 linhas × 13 colunas).
[PASSOU] Dicionário carregado (1,593 linhas × 5 colunas lidas).


## 2. Schema

Compara colunas (nomes e ordem) e tipos com o documentado em `dados/brutos/README.md`. Uma coluna renomeada ou ausente aparece como diferença exata entre os conjuntos.

In [3]:
SECAO = "2. Schema"
ESPERADAS_BASE = ["product_id", "description", "unit", "price", "location", "source", "collected_at",
                  "und_val", "und_med", "preco_kg", "codg_item", "desc_ref", "match_score"]
faltando = [c for c in ESPERADAS_BASE if c not in df.columns]
extras = [c for c in df.columns if c not in ESPERADAS_BASE]
if faltando or extras:
    registrar("BLOQUEANTE", SECAO, f"Base: colunas ausentes {faltando}, inesperadas {extras}.")
elif list(df.columns) != ESPERADAS_BASE:
    registrar("WARNING", SECAO, "Base: mesmas colunas, mas em ordem diferente da documentada.")
else:
    registrar("PASSOU", SECAO, "Base: as 13 colunas documentadas, nome e ordem.")

tipos_esperados = {
    "product_id": pd.api.types.is_integer_dtype,
    "description": pd.api.types.is_string_dtype,
    "price": pd.api.types.is_float_dtype,
    "location": pd.api.types.is_string_dtype,
    "source": pd.api.types.is_string_dtype,
    "und_val": pd.api.types.is_float_dtype,
    "preco_kg": pd.api.types.is_float_dtype,
    "match_score": pd.api.types.is_float_dtype,
}
tipos_errados = {c: str(df[c].dtype) for c, ok in tipos_esperados.items() if c in df.columns and not ok(df[c])}
if tipos_errados:
    registrar("BLOQUEANTE", SECAO, f"Base: tipos fora do esperado {tipos_errados}.")
else:
    registrar("PASSOU", SECAO, "Base: tipos corretos nas colunas numéricas e de texto verificadas.")

datas = pd.to_datetime(df["collected_at"], errors="coerce")
if datas.isna().any():
    registrar("BLOQUEANTE", SECAO, f"`collected_at`: {int(datas.isna().sum())} valores não interpretáveis como data.")
else:
    registrar("PASSOU", SECAO, "`collected_at` interpretável como data em 100% dos registros (armazenada como texto no CSV).")

ESPERADAS_DIC = ["desc_alim", "codg_item1", "cod_padrao"]
sem_nome = [c for c in dic.columns if str(c).startswith("Unnamed")]
if any(c not in dic.columns for c in ESPERADAS_DIC):
    registrar("BLOQUEANTE", SECAO, f"Dicionário: faltam colunas {[c for c in ESPERADAS_DIC if c not in dic.columns]}.")
else:
    registrar("PASSOU", SECAO, "Dicionário: `desc_alim`, `codg_item1` e `cod_padrao` presentes.")
if sem_nome:
    vazias = [c for c in sem_nome if dic[c].isna().all()]
    registrar("WARNING", SECAO, f"Dicionário: {len(sem_nome)} colunas sem nome no CSV ({sem_nome}); {len(vazias)} totalmente vazias — ruído do arquivo original (README), ignoradas no pipeline.")

pd.DataFrame({"dtype": df.dtypes.astype(str)})

[PASSOU] Base: as 13 colunas documentadas, nome e ordem.
[PASSOU] Base: tipos corretos nas colunas numéricas e de texto verificadas.
[PASSOU] `collected_at` interpretável como data em 100% dos registros (armazenada como texto no CSV).
[PASSOU] Dicionário: `desc_alim`, `codg_item1` e `cod_padrao` presentes.
[WARNING] Dicionário: 2 colunas sem nome no CSV (['Unnamed: 3', 'Unnamed: 4']); 2 totalmente vazias — ruído do arquivo original (README), ignoradas no pipeline.


,dtype
product_id,int64
description,str
unit,str
price,float64
location,str
source,str
collected_at,str
und_val,float64
und_med,str
preco_kg,float64


## 3. Completude

Taxa de nulos e de strings vazias por coluna, agrupadas pelo papel definido no ADR-0011. As colunas de **texto** são as críticas: nulo ou vazio em `description` impede formar o par.

In [4]:
SECAO = "3. Completude"
PAPEL = {
    "description": "texto (modelo)", "product_id": "rastreio", "source": "rastreio", "location": "rastreio",
    "collected_at": "rastreio", "unit": "unidade", "und_val": "unidade", "und_med": "unidade",
    "price": "fora do matching", "preco_kg": "fora do matching",
    "codg_item": "matcher ALTA", "desc_ref": "matcher ALTA", "match_score": "matcher ALTA",
}
vazios = {c: int((df[c].astype("string").str.strip() == "").sum()) for c in df.columns}
completude = pd.DataFrame({
    "papel": pd.Series(PAPEL),
    "nulos_%": (df.isna().mean() * 100).round(2),
    "vazios_str": pd.Series(vazios),
}).loc[df.columns]
display(completude)

for col in ["description", "source", "location", "product_id"]:
    n = int(df[col].isna().sum() + vazios[col])
    if n:
        registrar("BLOQUEANTE", SECAO, f"`{col}` (texto/rastreio): {n} registros nulos ou vazios.")
    else:
        registrar("PASSOU", SECAO, f"`{col}`: 100% preenchida.")

for col in ["price", "collected_at"]:
    n = int(df[col].isna().sum())
    registrar("PASSOU" if n == 0 else "WARNING", SECAO, f"`{col}`: {n} nulos (campo fora do matching).")

pct_unit = df["unit"].isna().mean() * 100
pct_und = df["und_med"].isna().mean() * 100
registrar("WARNING", SECAO,
          f"Metadado de unidade incompleto: `unit` {pct_unit:.1f}% nulo, `und_val`/`und_med` {pct_und:.1f}% nulo. "
          "A supervisão distante (spec 001, RF2) usa unidade normalizada — esses registros ficam sem esse sinal.")

pct_cod = df["codg_item"].isna().mean() * 100
registrar("WARNING", SECAO,
          f"`codg_item`/`desc_ref` ausentes em {pct_cod:.1f}% dos registros: o baseline 'ALTA atual' (ADR-0011) "
          "não tem predição nesses casos e deve ser reportado nas duas versões (condicionada e total).")

,papel,nulos_%,vazios_str
product_id,rastreio,0.00,0
description,texto (modelo),0.00,0
unit,unidade,2.04,0
price,fora do matching,0.00,0
location,rastreio,0.00,0
source,rastreio,0.00,0
collected_at,rastreio,0.00,0
und_val,unidade,5.80,0
und_med,unidade,5.80,0
preco_kg,fora do matching,5.80,0


[PASSOU] `description`: 100% preenchida.
[PASSOU] `source`: 100% preenchida.
[PASSOU] `location`: 100% preenchida.
[PASSOU] `product_id`: 100% preenchida.
[PASSOU] `price`: 0 nulos (campo fora do matching).
[PASSOU] `collected_at`: 0 nulos (campo fora do matching).
[WARNING] Metadado de unidade incompleto: `unit` 2.0% nulo, `und_val`/`und_med` 5.8% nulo. A supervisão distante (spec 001, RF2) usa unidade normalizada — esses registros ficam sem esse sinal.
[WARNING] `codg_item`/`desc_ref` ausentes em 19.2% dos registros: o baseline 'ALTA atual' (ADR-0011) não tem predição nesses casos e deve ser reportado nas duas versões (condicionada e total).


## 4. Duplicatas

Três níveis: (a) linhas 100% idênticas; (b) "quase exatas" — mesma descrição normalizada (maiúsculas, sem acento) na **mesma loja**, mas com algum outro campo diferente, o que pode indicar erro de coleta; (c) unicidade de `(source, product_id)`, a chave que o ADR-0011 usa para identificar o registro de origem de cada par.

In [5]:
SECAO = "4. Duplicatas"
n = len(df)
dup_exatas = df.duplicated(keep="first")
n_exatas = int(dup_exatas.sum())
print(f"Duplicatas exatas (excedentes): {n_exatas:,} ({n_exatas / n:.2%})")
display(df[dup_exatas]["source"].value_counts().rename("excedentes").to_frame())

df["_desc_norm"] = df["description"].map(normalizar)
df_unica = df.loc[~dup_exatas]
mascara_quase = df_unica.duplicated(["_desc_norm", "source"], keep=False)
quase = df_unica[mascara_quase]
n_grupos_quase = quase.groupby(["_desc_norm", "source"]).ngroups
print(f"\nQuase-duplicatas (mesma descrição normalizada + loja, não idênticas): {len(quase):,} linhas em {n_grupos_quase:,} grupos")
mesmo_preco = quase.groupby(["_desc_norm", "source"])["price"].nunique().eq(1).sum()
print(f"  grupos com o mesmo preço: {mesmo_preco}; com preços diferentes: {n_grupos_quase - mesmo_preco}")
display(quase.sort_values(["source", "_desc_norm"]).groupby("source").head(1).sample(
    min(5, quase["source"].nunique()), random_state=SEED)[["product_id", "description", "price", "source", "location"]])

chave_duplicada = int(df_unica.duplicated(["source", "product_id"]).sum())

if n_exatas:
    registrar("WARNING", SECAO,
              f"{n_exatas:,} linhas duplicadas exatas ({n_exatas / n:.1%}), idênticas até no timestamp. "
              "Os 30.259 registros brutos as incluem; a EDA descreve a base deduplicada (ADR-0012). A deduplicação é feita pelo script 01 (ADR-0012), antes de qualquer etapa seguinte: "
              "a mesma descrição em treino e teste seria vazamento.")
else:
    registrar("PASSOU", SECAO, "Sem linhas duplicadas exatas.")
registrar("WARNING" if n_grupos_quase else "PASSOU", SECAO,
          f"{n_grupos_quase} grupos de quase-duplicatas (mesma descrição normalizada + loja, campos distintos), "
          "possível erro de coleta ou o mesmo item em preços/ids diferentes; não removidos aqui.")
if chave_duplicada:
    registrar("BLOQUEANTE", SECAO, f"`(source, product_id)` não é único após remover duplicatas exatas ({chave_duplicada} repetições): "
              "não identifica o registro de origem do par.")
else:
    registrar("PASSOU", SECAO, "`(source, product_id)` é único após remover as duplicatas exatas — chave de rastreio do ADR-0011 válida.")

Duplicatas exatas (excedentes): 1,129 (3.73%)


,excedentes
source,
extra_api,1028
chegou_a_feira_api,100
guedes_api,1



Quase-duplicatas (mesma descrição normalizada + loja, não idênticas): 394 linhas em 185 grupos
  grupos com o mesmo preço: 36; com preços diferentes: 149


,product_id,description,price,source,location
1736,5622,Manteiga Président Com sal 200g,14.99,atacadao_api,"CE, Fortaleza, Papicu"
23238,39689,ACUCAR ALEGRE CRISTAL 1 KG,3.49,guedes_api,"PB, Patos, Centro"
959,16400,Energetico Tnt Energy Drink Maca Verde Zero Lata 269ml,7.99,ajuba_api,"PB, João Pessoa, Pedro Gondim"
24995,3251,Bacon Manta 300g,10.19,pinheiro_api,"CE, Limoeiro do Norte, Socorro"
4205,5129,Atum Ralado ao Molho de Tomate Gomes da Costa 170g,9.99,chegou_a_feira_api,"PB, João Pessoa, Pedro Gondim"


[WARNING] 1,129 linhas duplicadas exatas (3.7%), idênticas até no timestamp. Os 30.259 registros brutos as incluem; a EDA descreve a base deduplicada (ADR-0012). A deduplicação é feita pelo script 01 (ADR-0012), antes de qualquer etapa seguinte: a mesma descrição em treino e teste seria vazamento.
[WARNING] 185 grupos de quase-duplicatas (mesma descrição normalizada + loja, campos distintos), possível erro de coleta ou o mesmo item em preços/ids diferentes; não removidos aqui.
[PASSOU] `(source, product_id)` é único após remover as duplicatas exatas — chave de rastreio do ADR-0011 válida.


## 5. Consistência de domínio

- **Estado/loja:** só PB/CE/PI (escopo do projeto).
- **Preço:** negativo ou zero. Preço não entra no modelo, então é defeito de dado, não de matching.
- **Data:** dentro do período de coleta documentado — o README descreve uma **coleta pontual de 10/09/2026**.
- **Unidade:** vocabulário de `und_med` e grafias de `unit`.
- **Texto:** espaços nas pontas e duplos em `description`.

In [6]:
SECAO = "5. Domínio"
ESTADOS_OK = {"PB", "CE", "PI"}
estado = df["location"].map(lambda s: s.split(",", 1)[0].strip() if "," in s else "")
fora = estado[~estado.isin(ESTADOS_OK)]
if len(fora):
    registrar("BLOQUEANTE", SECAO, f"{len(fora)} registros com estado fora de PB/CE/PI: {fora.value_counts().to_dict()}.")
else:
    registrar("PASSOU", SECAO, f"Estados somente PB/CE/PI ({estado.value_counts().to_dict()}).")
registrar("INFO", SECAO, f"{df['location'].nunique()} locais distintos; {df['source'].nunique()} redes (`source`).")

neg = int((df["price"] < 0).sum())
zero = int((df["price"] == 0).sum())
registrar("PASSOU" if neg == 0 else "WARNING", SECAO, f"Preços negativos: {neg}.")
registrar("WARNING" if zero else "PASSOU", SECAO,
          f"Preços iguais a zero: {zero} (irrelevante para o matching; não removido).")
registrar("INFO", SECAO, f"Preço: mediana {df['price'].median():.2f}, p99 {df['price'].quantile(.99):.2f}, máx {df['price'].max():.2f}.")
display(df.loc[df["price"] == 0, ["description", "price", "source", "location"]])

datas = pd.to_datetime(df["collected_at"])
dias = sorted(datas.dt.date.unique())
PERIODO_ESPERADO = (pd.Timestamp("2026-09-10").date(), pd.Timestamp("2026-09-10").date())
fora_periodo = int(((datas.dt.date < PERIODO_ESPERADO[0]) | (datas.dt.date > PERIODO_ESPERADO[1])).sum())
registrar("PASSOU" if fora_periodo == 0 else "BLOQUEANTE", SECAO,
          f"Datas fora do período esperado {PERIODO_ESPERADO[0]}: {fora_periodo}. Dias presentes: {[str(d) for d in dias]}.")
registrar("WARNING", SECAO,
          "Toda a base vem de um único dia de coleta: sem série temporal, e não representa o histórico do ALTA — limitação do TCC.")

print("\nund_med:", df["und_med"].value_counts(dropna=False).to_dict())
registrar("INFO", SECAO, f"`unit` tem {df['unit'].nunique()} grafias distintas (ex.: {df['unit'].value_counts().head(6).to_dict()}).")

esp_pontas = int((df["description"] != df["description"].str.strip()).sum())
esp_duplos = int(df["description"].str.contains(r"\s{2,}", regex=True).sum())
registrar("INFO", SECAO, f"`description` com espaços nas pontas: {esp_pontas}; com espaços duplos: {esp_duplos} (ruído de PDV, não corrigido).")

[PASSOU] Estados somente PB/CE/PI ({'PI': 10909, 'PB': 10387, 'CE': 8963}).
[INFO] 7 locais distintos; 8 redes (`source`).
[PASSOU] Preços negativos: 0.
[WARNING] Preços iguais a zero: 1 (irrelevante para o matching; não removido).
[INFO] Preço: mediana 12.19, p99 213.24, máx 3239.90.


,description,price,source,location
2233,Fórmula Infantil NAN Comfor HMO de 6 a 12 meses 800g,0.0,atacadao_api,"CE, Fortaleza, Papicu"


[PASSOU] Datas fora do período esperado 2026-09-10: 0. Dias presentes: ['2026-09-10'].
[WARNING] Toda a base vem de um único dia de coleta: sem série temporal, e não representa o histórico do ALTA — limitação do TCC.

und_med: {'G': 17369, 'ML': 7276, 'KG': 2197, nan: 1756, 'L': 1661}
[INFO] `unit` tem 38 grafias distintas (ex.: {'UN': 10964, 'PC': 6813, 'g': 6397, 'ml': 1476, 'UND': 1062, 'KG': 983}).
[INFO] `description` com espaços nas pontas: 1058; com espaços duplos: 582 (ruído de PDV, não corrigido).


## 6. Integridade referencial e cobertura do dicionário

**Estrutural (ADR-0011, uso (d)).** `codg_item` aponta para o dicionário? `desc_ref` é coerente com ele? Só agregados.

**Cobertura "por categoria".** Não há categoria utilizável: `codg_item` é restrito pelo ADR-0010/0011 e o significado de `cod_padrao` não foi confirmado (o `main.tex` decidiu não usá-lo). No lugar, três **proxies lexicais** de quantos registros têm ao menos um candidato plausível no dicionário, do mais conservador ao mais permissivo. São aproximações, não cobertura real — esta só sai da blocagem (tarefa 1.2).

In [7]:
SECAO = "6. Referencial"
preenchido = df["codg_item"].notna()
existe = df.loc[preenchido, "codg_item"].isin(dic["codg_item1"])
registrar("PASSOU" if existe.all() else "WARNING", SECAO,
          f"`codg_item` existe no dicionário em {existe.mean():.1%} dos {int(preenchido.sum()):,} registros preenchidos.")
mapa = dic.set_index("codg_item1")["desc_alim"].map(normalizar)
coerente = (df.loc[preenchido, "desc_ref"].map(normalizar) == df.loc[preenchido, "codg_item"].map(mapa)).mean()
registrar("PASSOU" if coerente == 1 else "WARNING", SECAO, f"`desc_ref` coerente com o `desc_alim` do código em {coerente:.1%} dos preenchidos.")

TOK = re.compile(r"[A-Z0-9]+")
desc_norm = df["description"].map(normalizar)
frases = sorted({normalizar(x) for x in dic["desc_alim"]}, key=len, reverse=True)
padrao_frase = re.compile(r"^(?:" + "|".join(re.escape(f) for f in frases) + r")\b")
primeiros = {(TOK.findall(normalizar(x)) or [""])[0] for x in dic["desc_alim"]}
vocab_dic = {t for x in dic["desc_alim"] for t in TOK.findall(normalizar(x))}

proxies = pd.Series({
    "frase canônica completa no início da descrição": desc_norm.map(lambda s: bool(padrao_frase.match(s))).mean(),
    "1º token igual ao 1º token de alguma entrada": desc_norm.map(lambda s: (TOK.findall(s) or [""])[0] in primeiros).mean(),
    "≥ 1 token do vocabulário canônico presente": desc_norm.map(lambda s: bool(set(TOK.findall(s)) & vocab_dic)).mean(),
}).map("{:.1%}".format)
display(proxies.to_frame("registros com candidato plausível"))
registrar("INFO", SECAO,
          "Cobertura lexical por proxy: " + "; ".join(f"{k}: {v}" for k, v in proxies.items()) +
          ". Faixa, não valor real; a cobertura efetiva depende da blocagem (tarefa 1.2).")

[PASSOU] `codg_item` existe no dicionário em 100.0% dos 24,445 registros preenchidos.
[PASSOU] `desc_ref` coerente com o `desc_alim` do código em 100.0% dos preenchidos.


,registros com candidato plausível
frase canônica completa no início da descrição,43.0%
1º token igual ao 1º token de alguma entrada,75.3%
≥ 1 token do vocabulário canônico presente,93.3%


[INFO] Cobertura lexical por proxy: frase canônica completa no início da descrição: 43.0%; 1º token igual ao 1º token de alguma entrada: 75.3%; ≥ 1 token do vocabulário canônico presente: 93.3%. Faixa, não valor real; a cobertura efetiva depende da blocagem (tarefa 1.2).


## 7. Dicionário canônico isolado

Duplicatas internas (exatas e normalizadas), campos obrigatórios, unicidade de código e consistência de granularidade. O `main.tex` e o ADR-0007 assumem entradas **genéricas**: sem marca e sem tamanho. Se aparecer entrada com tamanho/unidade, o argumento "tamanhos diferentes são match" (protocolo §3.4) precisa ser revisto.

In [8]:
SECAO = "7. Dicionário"
obrig = ["desc_alim", "codg_item1", "cod_padrao"]
nulos_dic = {c: int(dic[c].isna().sum()) for c in obrig}
vazias_dic = int((dic["desc_alim"].astype("string").str.strip() == "").sum())
registrar("PASSOU" if not any(nulos_dic.values()) and vazias_dic == 0 else "BLOQUEANTE", SECAO,
          f"Campos obrigatórios: nulos {nulos_dic}, `desc_alim` vazia {vazias_dic}.")

d_exata = int(dic["desc_alim"].duplicated().sum())
d_norm = int(dic["desc_alim"].map(normalizar).duplicated().sum())
d_cod = int(dic["codg_item1"].duplicated().sum())
registrar("PASSOU" if (d_exata + d_norm + d_cod) == 0 else "BLOQUEANTE", SECAO,
          f"Duplicatas internas: `desc_alim` exata {d_exata}, normalizada {d_norm}, `codg_item1` {d_cod}.")

padrao_tamanho = re.compile(r"\b\d+([.,]\d+)?\s?(KG|G|GR|ML|L|LT|UN|UND|MG)\b", re.IGNORECASE)
com_tamanho = dic[dic["desc_alim"].str.contains(padrao_tamanho)]
com_digito = dic[dic["desc_alim"].str.contains(r"\d")]
registrar("PASSOU" if len(com_tamanho) == 0 else "BLOQUEANTE", SECAO,
          f"Entradas com tamanho/unidade no nome: {len(com_tamanho)} — granularidade genérica "
          f"{'consistente' if len(com_tamanho) == 0 else 'INCONSISTENTE'} com o ADR-0007/protocolo §3.4.")
display(com_digito[["desc_alim"]])
registrar("INFO", SECAO, f"{len(com_digito)} entradas têm dígitos, todas como variante descritiva (ex.: '3 GRÃOS'), não tamanho.")

dist = dic["cod_padrao"].value_counts().sort_index()
dominante = dist.max() / len(dic)
registrar("WARNING", SECAO,
          f"`cod_padrao`: {dist.idxmax()} concentra {dominante:.1%} das entradas ({dist.to_dict()}); significado não confirmado com o ALTA — "
          "inadequado como categoria para estratificação, como o main.tex já assume.")
registrar("INFO", SECAO, f"Comprimento de `desc_alim`: mediana {dic['desc_alim'].str.len().median():.0f} caracteres, "
          f"{dic['desc_alim'].map(lambda s: len(s.split())).median():.0f} palavras.")

[PASSOU] Campos obrigatórios: nulos {'desc_alim': 0, 'codg_item1': 0, 'cod_padrao': 0}, `desc_alim` vazia 0.
[PASSOU] Duplicatas internas: `desc_alim` exata 0, normalizada 0, `codg_item1` 0.
[PASSOU] Entradas com tamanho/unidade no nome: 0 — granularidade genérica consistente com o ADR-0007/protocolo §3.4.


/var/folders/jk/yh8n66d900g0nv07dxyxh8ym0000gn/T/ipykernel_8320/3976199347.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  com_tamanho = dic[dic["desc_alim"].str.contains(padrao_tamanho)]


,desc_alim
3,"ARROZ, 3 GRAOS"
4,"ARROZ, 7 GRAOS"
987,"BEBIDA REFRIGERANTE, TIPO H2OH®"


[INFO] 3 entradas têm dígitos, todas como variante descritiva (ex.: '3 GRÃOS'), não tamanho.
[WARNING] `cod_padrao`: 99 concentra 82.7% das entradas ({1: 38, 2: 58, 3: 43, 4: 46, 5: 4, 6: 2, 7: 65, 8: 6, 9: 14, 99: 1317}); significado não confirmado com o ALTA — inadequado como categoria para estratificação, como o main.tex já assume.
[INFO] Comprimento de `desc_alim`: mediana 16 caracteres, 3 palavras.


## 8. Veredicto

Consolidação automática de tudo que foi registrado acima. **Nada foi corrigido neste notebook**; o que está em `BLOQUEANTE` precisa de correção antes da EDA/blocagem, e o que está em `WARNING` deve ser citado no TCC como limitação dos dados.

In [9]:
veredito = pd.DataFrame(VEREDITOS)
ordem = ["BLOQUEANTE", "WARNING", "INFO", "PASSOU"]
resumo = veredito["nivel"].value_counts().reindex(ordem, fill_value=0)
display(resumo.to_frame("verificações"))

for nivel in ordem:
    itens = veredito[veredito["nivel"] == nivel]
    print(f"\n=== {nivel} ({len(itens)}) ===")
    for _, linha in itens.iterrows():
        print(f"- [{linha['secao']}] {linha['mensagem']}")

n_bloq = int(resumo["BLOQUEANTE"])
print("\n" + "=" * 70)
print("VEREDICTO:", "APTO para EDA/blocagem (sem erros bloqueantes)." if n_bloq == 0
      else f"NÃO APTO: {n_bloq} erro(s) bloqueante(s) a corrigir antes de seguir.")

,verificações
nivel,
BLOQUEANTE,0
WARNING,8
INFO,7
PASSOU,21



=== BLOQUEANTE (0) ===

=== WARNING (8) ===
- [2. Schema] Dicionário: 2 colunas sem nome no CSV (['Unnamed: 3', 'Unnamed: 4']); 2 totalmente vazias — ruído do arquivo original (README), ignoradas no pipeline.
- [3. Completude] Metadado de unidade incompleto: `unit` 2.0% nulo, `und_val`/`und_med` 5.8% nulo. A supervisão distante (spec 001, RF2) usa unidade normalizada — esses registros ficam sem esse sinal.
- [3. Completude] `codg_item`/`desc_ref` ausentes em 19.2% dos registros: o baseline 'ALTA atual' (ADR-0011) não tem predição nesses casos e deve ser reportado nas duas versões (condicionada e total).
- [4. Duplicatas] 1,129 linhas duplicadas exatas (3.7%), idênticas até no timestamp. Os 30.259 registros brutos as incluem; a EDA descreve a base deduplicada (ADR-0012). A deduplicação é feita pelo script 01 (ADR-0012), antes de qualquer etapa seguinte: a mesma descrição em treino e teste seria vazamento.
- [4. Duplicatas] 185 grupos de quase-duplicatas (mesma descrição normalizada + l